### Supplement 4: Orchestrator and workers (`4-orchestrator.py`)

Write a blog post with three roles: An orchestrator plans the sections, a worker writes each section, and a reviewer polishes the whole post.

**The key point:** The first LLM call decides how many steps there will be. It returns a plan with a list of sections, and your code loops over that list, making one worker call per section. In prompt chaining the steps are fixed in the code; here the model plans the work, but your code still runs the loop.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
topic, target_length, style
   │
   ▼
Orchestrator   get_plan          → OrchestratorPlan
   │           topic_analysis, target_audience, sections[...]
   ▼
Workers        for section in plan.sections:
   │               write_section → SectionContent    One call each
   ▼
Reviewer       review_post       → ReviewFeedback
               cohesion_score, suggested_edits, final_version
</pre>

#### 1. Setup
Same as Supplement 1: `load_dotenv()` puts `OPENAI_API_KEY` into `os.environ`, and `OpenAI()` reads it from there.

In [1]:
from typing import List, Dict
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
model = "gpt-5-nano"

#### 2. The data models
- `SubTask`: One section for a worker to write. The orchestrator fills in a list of these.
- `OrchestratorPlan`: The plan. `sections` is a `List[SubTask]`, so the number of sections is up to the model.
- `SectionContent`: What a worker returns for one section.
- `ReviewFeedback`: The reviewer's verdict, including `final_version`, the whole polished post.

In [2]:
# Define Data Models

class SubTask(BaseModel):
    """Blog section task defined by orchestrator"""

    section_type: str = Field(description="Type of blog section to write")
    description: str = Field(description="What this section should cover")
    style_guide: str = Field(description="Writing style for this section")
    target_length: int = Field(description="Target word count for this section")


class OrchestratorPlan(BaseModel):
    """Orchestrator's blog structure and tasks"""

    topic_analysis: str = Field(description="Analysis of the blog topic")
    target_audience: str = Field(description="Intended audience for the blog")
    sections: List[SubTask] = Field(description="List of sections to write")


class SectionContent(BaseModel):
    """Content written by a worker"""

    content: str = Field(description="Written content for the section")
    key_points: List[str] = Field(description="Main points covered")


class SuggestedEdits(BaseModel):
    """Suggested edits for a section"""

    section_name: str = Field(description="Name of the section")
    suggested_edit: str = Field(description="Suggested edit")


class ReviewFeedback(BaseModel):
    """Final review and suggestions"""

    cohesion_score: float = Field(description="How well sections flow together (0-1)")
    suggested_edits: List[SuggestedEdits] = Field(
        description="Suggested edits by section"
    )
    final_version: str = Field(description="Complete, polished blog post")

#### 3. The prompts
Three prompt templates. Each `{topic}`-style slot is filled in later by `.format(...)`, which replaces `{name}` with the value passed as `name=...`.

The prompts describe a Markdown layout ("# Analysis", "# Content", ...), but `response_format` forces each reply to be JSON that matches its class. The schema wins, and the layout only hints at what to write.

In [3]:
# Define Prompts

ORCHESTRATOR_PROMPT = """
Analyze this blog topic and break it down into logical sections.

Topic: {topic}
Target Length: {target_length} words
Style: {style}

Return your response in this format:

# Analysis
Analyze the topic and explain how it should be structured.
Consider the narrative flow and how sections will work together.

# Target Audience
Define the target audience and their interests/needs.

# Sections
## Section 1
- Type: section_type
- Description: what this section should cover
- Style: writing style guidelines

[Additional sections as needed...]
"""

In [4]:
WORKER_PROMPT = """
Write a blog section based on:
Topic: {topic}
Section Type: {section_type}
Section Goal: {description}
Style Guide: {style_guide}
Target Length: {target_length} words

Previous sections, for context (don't repeat them):
{previous_sections}

Return your response in this format:

# Content
[Your section content here, following the style guide]

# Key Points
- Main point 1
- Main point 2
[Additional points as needed...]
"""

In [5]:
REVIEWER_PROMPT = """
Review this blog post for cohesion and flow:

Topic: {topic}
Target Audience: {audience}

Sections:
{sections}

Provide a cohesion score between 0.0 and 1.0, suggested edits for each section if needed, and a final polished version of the complete post.

The cohesion score should reflect how well the sections flow together, with 1.0 being perfect cohesion.
For suggested edits, focus on improving transitions and maintaining consistent tone across sections.
The final version should incorporate your suggested improvements into a polished, cohesive blog post.
"""

#### 4. The orchestrator class
`BlogOrchestrator` keeps every written section in `self.sections_content`, a dict from a numbered section name (like `"1. Introduction"`) to its `SectionContent`, so later workers and the reviewer can read them.
- `get_plan`: The orchestrator call. It returns an `OrchestratorPlan`.
- `write_section`: The worker call for one section. It sends along the sections written so far, or "This is the first section."
- `review_post`: The reviewer call. It joins all the written sections into one text and asks for a polished version.
- `write_blog`: Runs the three roles in order, with a `for` loop over `plan.sections` in the middle. `enumerate(plan.sections, start=1)` hands out a running number (1, 2, 3, ...) with each section.

Three fixes to the course code, made in `4-orchestrator.py` too:
- `WORKER_PROMPT` had no `{previous_sections}` or `{target_length}` slot, and `.format()` silently ignores values it has no slot for. So the worker never saw the earlier sections or the word count. The prompt now has both slots.
- Sections were stored under their name alone, and a dict keeps one value per key, so two sections with the same name would have overwritten each other. The key now starts with the section's number.
- `self.sections_content` was only created once, in `__init__`, so calling `write_blog` a second time on the same object would reuse the first blog's sections as context. `write_blog` now empties it first.

In [6]:
#Implement Orchestrator

class BlogOrchestrator:
    def __init__(self):
        self.sections_content = {}

    def get_plan(self, topic: str, target_length: int, style: str) -> OrchestratorPlan:
        """Get orchestrator's blog structure plan"""
        completion = client.chat.completions.parse(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": ORCHESTRATOR_PROMPT.format(
                        topic=topic, target_length=target_length, style=style
                    ),
                }
            ],
            response_format=OrchestratorPlan,
        )
        return completion.choices[0].message.parsed

    def write_section(self, topic: str, section: SubTask) -> SectionContent:
        """Worker: Write a specific blog section with context from previous sections.

        Args:
            topic: The main blog topic
            section: SubTask containing section details

        Returns:
            SectionContent: The written content and key points
        """
        # Create context from previously written sections
        previous_sections = "\n\n".join(
            [
                f"=== {section_type} ===\n{content.content}"
                for section_type, content in self.sections_content.items()
            ]
        )

        completion = client.chat.completions.parse(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": WORKER_PROMPT.format(
                        topic=topic,
                        section_type=section.section_type,
                        description=section.description,
                        style_guide=section.style_guide,
                        target_length=section.target_length,
                        previous_sections=previous_sections
                        if previous_sections
                        else "This is the first section.",
                    ),
                }
            ],
            response_format=SectionContent,
        )
        return completion.choices[0].message.parsed

    def review_post(self, topic: str, plan: OrchestratorPlan) -> ReviewFeedback:
        """Reviewer: Analyze and improve overall cohesion"""
        sections_text = "\n\n".join(
            [
                f"=== {section_type} ===\n{content.content}"
                for section_type, content in self.sections_content.items()
            ]
        )

        completion = client.chat.completions.parse(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": REVIEWER_PROMPT.format(
                        topic=topic,
                        audience=plan.target_audience,
                        sections=sections_text,
                    ),
                }
            ],
            response_format=ReviewFeedback,
        )
        return completion.choices[0].message.parsed

    def write_blog(
        self, topic: str, target_length: int = 1000, style: str = "informative"
    ) -> Dict:
        #Just in case the write_blog() is called a second time
        self.sections_content = {}
        
        """Process the entire blog writing task"""
        print(f"Starting blog writing process for: {topic}")

        # Get blog structure plan
        plan = self.get_plan(topic, target_length, style)
        print(f"Blog structure planned: {len(plan.sections)} sections")
        print(f"Blog structure planned: {plan.model_dump_json(indent=2)}")

        # Write each section
        for number, section in enumerate(plan.sections, start=1):
            print(f"Writing section: {section.section_type}")
            content = self.write_section(topic, section)
            self.sections_content[f"{number}. {section.section_type}"] = content

        # Review and polish
        print("Reviewing full blog post")
        review = self.review_post(topic, plan)

        return {"structure": plan, "sections": self.sections_content, "review": review}

#### 5. The orchestrator on its own: The plan
The cells below call `get_plan` once, so you can see the plan before the full run.

In [7]:
orchestrator = BlogOrchestrator()
topic = "The impact of AI on software development"

plan = orchestrator.get_plan(topic=topic, target_length=1200, style="technical but accessible")

In [8]:
print(type(plan))
print(plan.model_dump_json(indent=2))

<class '__main__.OrchestratorPlan'>
{
  "topic_analysis": "The blog topic explores how artificial intelligence (AI) is reshaping software development across the entire lifecycle, from idea to deployment and maintenance. The piece should balance practical guidance with a clear conceptual framework: definitions of AI in this context, a landscape of tools and techniques, how AI changes roles and workflows, concrete benefits and measurable outcomes, and the risks, governance, and future directions. A logical arc is essential—start with context and definitions, move through capability categories and organizational impact, address risks and governance, present real-world examples, and finish with actionable guidance for teams planning an AI-enabled strategy. The narrative should demystify AI without glossing over limitations (hallucinations, bias, data leakage, licensing), and offer frameworks (checklists, metrics, governance models) that teams can apply today. The target length is approxima

#### 6. One worker on its own
This writes the first section of that plan. Only `write_blog` stores sections in `self.sections_content`, so this call doesn't add anything there.

In [9]:
first_section = orchestrator.write_section(topic, plan.sections[0])

In [10]:
print(type(first_section))
print(first_section.model_dump_json(indent=2))

<class '__main__.SectionContent'>
{
  "content": "AI is reshaping how software is written and maintained. In this article, AI in dev denotes the broader use of AI across the development lifecycle—planning, coding, testing, deployment, and maintenance. Generative AI refers to models that produce new content—code, documentation, tests, or designs—from prompts. Code assistants are AI-enabled tools embedded in editors that offer real-time suggestions, snippets, and automated refactoring.\n\nThesis: AI augments human developers by handling repetitive tasks, surfacing patterns, and enabling rapid experimentation, while core decisions, system architecture, and responsible use stay with people. The scope is practical: we examine capabilities, limits, workflows, governance, and observable impact across teams and project sizes. We aim for a measured view—no hype, just evidence-relevant insight.\n\nRoadmap: this piece proceeds in five parts: 1) what AI can and cannot do in development; 2) common 

#### 7. The full run
`write_blog` does everything again from scratch: A new plan, one worker call per section, then the review. It takes a few minutes.

`__name__` is a variable Python sets to `"__main__"` in the file or notebook you run directly, so the `if __name__ == "__main__":` block runs here too.

In [11]:
#Example Usage

if __name__ == "__main__":
    orchestrator = BlogOrchestrator()

    # Example: Technical blog post
    topic = "The impact of AI on software development"
    result = orchestrator.write_blog(
        topic=topic, target_length=1200, style="technical but accessible"
    )

    print("\nFinal Blog Post:")
    print(result["review"].final_version)

    print("\nCohesion Score:", result["review"].cohesion_score)
    if result["review"].suggested_edits:
        for edit in result["review"].suggested_edits:
            print(f"Section: {edit.section_name}")
            print(f"Suggested Edit: {edit.suggested_edit}")

Starting blog writing process for: The impact of AI on software development
Blog structure planned: 6 sections
Blog structure planned: {
  "topic_analysis": "The blog will explore how AI changes software development across the full lifecycle, from ideation to deployment and maintenance. The narrative should distinguish AI as a powerful augmentation tool (assistive AI that speeds up tasks, improves consistency, and augments decision making) from AI-driven transformations that genuinely change workflows. The outline below structures the discussion around concrete impact areas (productivity, tooling, testing, security and governance), organizational implications (skills, teams, governance), and a practical roadmap for adoption. It also foregrounds risks (hallucinations, data privacy, licensing, bias) and mitigations, ensuring a balanced view that translates to actionable guidance for engineers and leaders. The target length guidance (approx. 1200 words) is distributed across six sections 